# Step 1: Pre-processing (01_Preprocessing)

This notebook demonstrates the preprocessing steps used to prepare the Rubik's Cube images for alignment and defect detection.

We investigate three preprocessing methods to handle lighting and color variability:
1. Grayscale conversion.
2. HSV Value (V) channel extraction.
3. LAB Lightness (L) channel extraction.

We then apply **CLAHE** (Contrast Limited Adaptive Histogram Equalization) to balance the lighting (especially around shadows) and a **Gaussian Blur** filter to reduce noise and reflections.

In [ ]:
%load_ext autoreload
%autoreload 2
import cv2
import matplotlib.pyplot as plt
import core_functions as cf

# Set matplotlib parameters for high-quality display
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Loading the Images
We load reference and inspection images from both the solved and scrambled pipelines.

In [ ]:
ref_solved = cf.load_image('data/solved_pipeline/ref_solved.jpg')
inspect_solved_rot = cf.load_image('data/solved_pipeline/inspect_solved_rotated_layer.jpg')

ref_scrambled = cf.load_image('data/scrambled_pipeline/ref_scrambled.jpg')
inspect_scrambled_shadow = cf.load_image('data/scrambled_pipeline/inspect_scrambled_shadow.jpg')

print("Images loaded successfully!")
print(f"Solved reference shape: {ref_solved.shape}")
print(f"Scrambled inspection (shadow) shape: {inspect_scrambled_shadow.shape}")

## 2. Evaluating Preprocessing Methods
We compare Gray, HSV (V channel), and LAB (L channel) preprocessing on a scrambled image with shadows. Notice how CLAHE on V and L channels separates illumination from color structure.

In [ ]:
methods = ['gray', 'hsv_v', 'lab_l']
for method in methods:
    processed = cf.preprocess_image(inspect_scrambled_shadow, method=method)
    cf.plot_before_after(
        inspect_scrambled_shadow, 
        processed, 
        title_orig="Original RGB (Scrambled Shadow)", 
        title_proc=f"Preprocessed (Method: {method})"
    )

## 3. Investigating the Effect of CLAHE on Shadows
Let's visually isolate the effect of CLAHE on the Value (V) channel of the scrambled shadow image. CLAHE prevents loss of detail in shadowed areas and dampens reflections from plastic stickers.

In [ ]:
hsv = cv2.cvtColor(inspect_scrambled_shadow, cv2.COLOR_RGB2HSV)
h, s, v = cv2.split(hsv)

# Preprocess V channel without CLAHE (just blur)
v_blurred = cv2.GaussianBlur(v, (5, 5), 0)

# Preprocess V channel with CLAHE + blur
clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
v_clahe = clahe.apply(v)
v_clahe_blurred = cv2.GaussianBlur(v_clahe, (5, 5), 0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
ax1.imshow(v_blurred, cmap='gray')
ax1.set_title("Value (V) Channel WITHOUT CLAHE (Shadow areas are very dark)")
ax1.axis('off')

ax2.imshow(v_clahe_blurred, cmap='gray')
ax2.set_title("Value (V) Channel WITH CLAHE (Contrast balanced, shadows neutralized)")
ax2.axis('off')
plt.show()

## Conclusion
The pre-processing pipeline consists of:
1. **Conversion to HSV/LAB** to decouple brightness from color channels.
2. **CLAHE** on the brightness channel to neutralize shadows and glare.
3. **Gaussian Blur** to filter out plastic reflection noise.

This establishes a robust foundation for feature extraction and alignment in the next notebook.